# PyPSA-Eur model inputs focusing on installed capacities

##  imports

In [ ]:
import pandas as pd
import pypsa
import matplotlib.pyplot as plt
import seaborn as sns
import os

## configurations

In [ ]:

year = 2030
network_path = "C:\\Users\\user\\Pictures\\Ember-Flexibility-Study\\results\\scenario_2030_flex_on\\networks\\flex_on_base_s_39__23H_2030.nc"
custom_ppl_path = "C:\\Users\\user\\Pictures\\Ember-Flexibility-Study\\data\\custom_ppl_2030.csv"
combined_chp_path = "C:\\Users\\user\\Downloads\\combined_chp_2030_clean.csv"
plt.style.use("bmh")
sns.set_style("darkgrid")
focus_countries = ["DE", "NL", "IT", "PL", "CZ", "GR"]
country_map = {
    'Albania': 'AL',
    'Austria': 'AT',
    'Bosnia and Herzegovina': 'BA',
    'Belgium': 'BE',
    'Bulgaria': 'BG',
    'Switzerland': 'CH',
    'Cyprus': 'CY',
    'Czechia': 'CZ',
    'Germany': 'DE',
    'Denmark': 'DK',
    'Estonia': 'EE',
    'Spain': 'ES',
    'Finland': 'FI',
    'France': 'FR',
    'United Kingdom': 'GB',
    'Greece': 'GR',
    'Croatia': 'HR',
    'Hungary': 'HU',
    'Ireland': 'IE',
    'Italy': 'IT',
    'Lithuania': 'LT',
    'Luxembourg': 'LU',
    'Latvia': 'LV',
    'Moldova': 'MD',
    'Montenegro': 'ME',
    'North Macedonia': 'MK',
    'Malta': 'MT',
    'Netherlands': 'NL',
    'Norway': 'NO',
    'Poland': 'PL',
    'Portugal': 'PT',
    'Romania': 'RO',
    'Serbia': 'RS',
    'Sweden': 'SE',
    'Slovenia': 'SI',
    'Slovakia': 'SK',
    'Turkey': 'TR',
    'Ukraine': 'UA'
}

## Load Data

In [ ]:
n = pypsa.Network(network_path)
custom_ppl = pd.read_csv(custom_ppl_path)
combined_chp = pd.read_csv(combined_chp_path)

## Process Ember Capacities

In [ ]:
def process_input_capacities():
    print("Processing input capacities from CSVs for 2030")
    custom_ppl['DateIn'] = pd.to_numeric(custom_ppl['DateIn'], errors='coerce')
    custom_ppl['DateOut'] = pd.to_numeric(custom_ppl['DateOut'], errors='coerce')
    active_custom = custom_ppl[
        (custom_ppl['DateIn'] <= year) &
        ((custom_ppl['DateOut'] >= year) | custom_ppl['DateOut'].isna())
    ]
    wind_techs = ['onwind', 'offwind-ac', 'offwind-dc', 'offwind-float', 'onshore', 'offshore']
    active_custom.loc[active_custom['Technology'].str.strip().str.lower().isin(wind_techs), 'Fueltype'] = 'Wind'
    battery_techs = ['battery', 'home battery']
    active_custom.loc[active_custom['Technology'].str.strip().str.lower().isin(battery_techs), 'Fueltype'] = 'battery'
    custom_grouped = active_custom.groupby(['Country', 'Fueltype'])['Capacity'].sum().unstack(fill_value=0)
    combined_chp['DateIn'] = pd.to_numeric(combined_chp['DateIn'], errors='coerce')
    combined_chp['DateOut'] = pd.to_numeric(combined_chp['DateOut'], errors='coerce')
    active_chp = combined_chp[
        (combined_chp['DateIn'] <= year) &
        ((combined_chp['DateOut'] >= year) | combined_chp['DateOut'].isna())
    ]
    chp_grouped = active_chp.groupby(['bus', 'carrier'])['p_nom'].sum().unstack(fill_value=0)
    chp_grouped.index = chp_grouped.index.map(lambda x: country_map.get(x, x)) 
    chp_grouped.index.name = 'Country'
    combined_cap = custom_grouped.add(chp_grouped, fill_value=0)
    combined_cap['Coal'] = combined_cap.get('Hard Coal', 0) + combined_cap.get('Lignite', 0) + combined_cap.get('hard coal', 0) + combined_cap.get('lignite', 0) + combined_cap.get('urban central coal CHP', 0) + combined_cap.get('urban central lignite CHP', 0)
    combined_cap['Gas'] = combined_cap.get('Natural Gas', 0) + combined_cap.get('gas', 0) + combined_cap.get('urban central gas CHP', 0) + combined_cap.get('urban central gas CHP CC', 0)
    combined_cap['Nuclear'] = combined_cap.get('Nuclear', 0) + combined_cap.get('nuclear', 0)
    combined_cap['Hydro'] = combined_cap.get('Hydro', 0) + combined_cap.get('ror', 0) + combined_cap.get('PHS', 0)
    combined_cap['Wind'] = combined_cap.get('Wind', 0)
    combined_cap['Solar'] = combined_cap.get('solar', 0) + combined_cap.get('solar-hsat', 0) + combined_cap.get('Solar', 0)
    combined_cap['Battery'] = combined_cap.get('battery', 0)
    drop_cols = ['Hard Coal', 'Lignite', 'Natural Gas', 'hard coal', 'lignite', 'gas', 'nuclear', 'ror', 'PHS', 'Oil', 'oil', 'Bioenergy', 'Other Fossil', 'urban central solid biomass CHP', 'urban central solid biomass CHP CC', 'urban central coal CHP', 'urban central lignite CHP', 'urban central gas CHP', 'urban central gas CHP CC', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float', 'solar', 'solar-hsat', 'battery']
    combined_cap = combined_cap.drop(columns=[col for col in drop_cols if col in combined_cap.columns])
    combined_cap = combined_cap.div(1000).round(2) 
    print(f"Processed input capacity data sample:\n{combined_cap.head().to_string()}")
    return combined_cap
input_capacity_processed = process_input_capacities()


## Process PyPSA Capacities

In [ ]:
def process_pypsa_capacity():
    def merge_and_replace(df, new_col, cols_to_merge, drop_original=True):
        available_cols = [col for col in cols_to_merge if col in df.columns]
        if not available_cols:
            return df
        df[new_col] = df[available_cols].sum(axis=1)
        if drop_original:
            df = df.drop(columns=available_cols, errors='ignore')
        return df
    conv_techs = ['OCGT', 'CCGT', 'coal', 'lignite', 'nuclear', 'oil', 'urban central gas CHP','urban central coal CHP', 'urban central lignite CHP']
    vres_tech = ['solar-hsat', 'solar', 'onwind', 'offwind-float', 'offwind-dc', 'offwind-ac', 'ror']
    bio_carriers = ['urban central solid biomass CHP', 'urban central solid biomass CHP CC']
    battery_carriers = ['battery charger', 'home battery charger']
    pypsa_country_tech = (
        n.generators.query("carrier in @vres_tech")
        .groupby(['bus', 'carrier'])['p_nom']
        .sum().unstack(fill_value=0).reset_index()
    )
    sto_country_tech = n.storage_units.groupby(['bus', 'carrier'])['p_nom'].sum().unstack(fill_value=0).reset_index()
    conv_links = n.links.query("carrier in @conv_techs")
    chp_links = conv_links[conv_links.carrier.str.contains('CHP', na=False)]
    non_chp_links = conv_links[~conv_links.carrier.str.contains('CHP', na=False)]
    if not non_chp_links.empty:
        is_electric_non = n.buses.loc[non_chp_links.bus1, 'carrier'] == 'AC'
        df = non_chp_links[is_electric_non.values]
        electrical_nom = df.p_nom.copy()
        electrical_nom *= df.efficiency
        non_chp_country_tech = (
            df.assign(electrical_nom=electrical_nom)
            .groupby(['bus1', 'carrier'])['electrical_nom']
            .sum().unstack(fill_value=0).reset_index()
            .rename(columns={'bus1': 'bus'})
        )
    else:
        non_chp_country_tech = pd.DataFrame()
    if not chp_links.empty:
        is_electric_chp = n.buses.loc[chp_links.bus1, 'carrier'] == 'AC'
        chp_country_tech = (
            chp_links[is_electric_chp.values]
            .assign(electrical_nom=lambda df: df.p_nom * df.efficiency)
            .groupby(['bus1', 'carrier'])['electrical_nom']
            .sum().unstack(fill_value=0).reset_index()
            .rename(columns={'bus1': 'bus'})
        )
    else:
        chp_country_tech = pd.DataFrame()
    conv_country_tech = pd.concat([non_chp_country_tech, chp_country_tech], ignore_index=True)
    bio_links = n.links[n.links['carrier'].isin(bio_carriers)].copy()
    if not bio_links.empty:
        bio_links['p_nom_used'] = bio_links['p_nom']
        bio_links['electrical_nom'] = bio_links['p_nom_used'] * bio_links['efficiency2']
        bio_links['bus'] = bio_links['bus2']
        bio_link_country_tech = (
            bio_links.groupby(['bus', 'carrier'])['electrical_nom']
            .sum().unstack(fill_value=0).reset_index()
        )
    else:
        bio_link_country_tech = pd.DataFrame()
    battery_links = n.links.query("carrier in @battery_carriers")
    if not battery_links.empty:
        battery_country_tech = (
            battery_links
            .groupby(['bus0', 'carrier'])['p_nom']
            .sum().unstack(fill_value=0).reset_index()
            .rename(columns={'bus0': 'bus'})
        )
    else:
        battery_country_tech = pd.DataFrame()
    pypsa_country_tech = pd.concat([pypsa_country_tech, sto_country_tech, conv_country_tech, bio_link_country_tech, battery_country_tech], ignore_index=True)
    pypsa_country_tech['country'] = pypsa_country_tech['bus'].str[:2]
    pypsa_country_tech_merged = pypsa_country_tech.groupby('country').sum(numeric_only=True).reset_index()
    wind_cols = [col for col in pypsa_country_tech_merged.columns if col.startswith("onwind") or col.startswith("offwind")]
    solar_cols = [col for col in pypsa_country_tech_merged.columns if col.startswith("solar")]
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Wind", wind_cols)
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Solar", solar_cols)
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Gas", ["CCGT", "OCGT", "urban central gas CHP", "urban central gas CHP CC"])
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Coal", ["coal", "lignite", "urban central coal CHP", "urban central lignite CHP"])
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Hydro", ["hydro", "ror", "PHS"])
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Nuclear", ["nuclear"])
    pypsa_country_tech_merged = merge_and_replace(pypsa_country_tech_merged, "Battery", ["battery", "home battery", "battery charger", "home battery charger"])
    drop_cols = ['Bioenergy', 'Other Fossil', 'oil', 'urban central solid biomass CHP', 'urban central solid biomass CHP CC']
    pypsa_country_tech_merged = pypsa_country_tech_merged.drop(columns=[col for col in drop_cols if col in pypsa_country_tech_merged.columns])
    pypsa_country_tech_merged = pypsa_country_tech_merged.set_index("country").div(1000).round(2) # MW to GW
    print(f"Processed PyPSA capacity data sample:\n{pypsa_country_tech_merged.head().to_string()}")
    return pypsa_country_tech_merged
pypsa_capacity_processed = process_pypsa_capacity()

## Merge ember and pypsa capacities

In [ ]:

combined = pd.concat([input_capacity_processed.add_suffix('_input'), pypsa_capacity_processed.add_suffix('_pypsa')], axis=1)
combined.to_csv("capacity_comparison_2030.csv")
print("Data saved to CSV files: input_capacity_2030.csv, pypsa_capacity_2030.csv, capacity_comparison_2030.csv")
print("Capacity Comparison:\n", combined.to_string())


## Plot  

In [ ]:

def plot_capacity_comparison_horizontal(countries, input_capacity, pypsa_capacity):
    print("Generating country-specific capacity comparison plot")
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    axes = axes.flatten()
  
    for idx, country in enumerate(countries):
        ax = axes[idx]
        input_row = input_capacity.loc[country] if country in input_capacity.index else None
        pypsa_row = pypsa_capacity.loc[country] if country in pypsa_capacity.index else None
      
        if input_row is not None and pypsa_row is not None:
            techs = sorted(set(input_row.index) | set(pypsa_row.index))
            techs = [t if t != "Other Renewables" else "Other RES" for t in techs]
          
            input_vals = [input_row.get(tech, 0) for tech in techs]
            pypsa_vals = [pypsa_row.get(tech, 0) for tech in techs]
          
            y = range(len(techs))
            height = 0.35
          
            ax.barh([i - height/2 for i in y], input_vals, height, label='Input CSVs', color='#13ce74')
            ax.barh([i + height/2 for i in y], pypsa_vals, height, label='PyPSA-Eur', color='#192238')
            ax.set_yticks(list(y))
            ax.set_yticklabels(techs)
            ax.set_title(f"{country} Capacity Comparison - 2030")
          
            if idx in [4, 5]:
                ax.set_xlabel("Capacity (GW)")
          
            if idx == 0:
                ax.legend(loc="upper right")
  
    for j in range(len(countries), len(axes)):
        axes[j].axis('off')
  
    fig.suptitle(f"Installed Capacity Comparison {year}", weight="bold")
    plt.tight_layout()
  
    output_path = f"results/validation_{year}/plots/country_capacity_plot.png"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.show()
plot_capacity_comparison_horizontal(focus_countries, input_capacity_processed, pypsa_capacity_processed)